# Practical 2 — Aged debtor review, Kestrel Supplies Ltd (tutor fallback notebook)

Model answer: find the data problems, clean, age at 31 March 2026, apply the provision matrix and draft the year-end journal. Totals are printed after each step as an audit trail.

**Run:** Runtime → Run all, then upload `sales_ledger.csv`.

In [ ]:
# Upload the CSV when running in Google Colab; otherwise read it from the current folder.
import pandas as pd
try:
    from google.colab import files
    uploaded = files.upload()          # choose sales_ledger.csv when prompted
except ImportError:
    pass

raw = pd.read_csv('sales_ledger.csv', dtype={'invoice_date': str})
print(len(raw), 'rows; gross outstanding', round((raw.amount - raw.amount_paid).sum(), 2))
raw.head()

## 1. Find the problems (do not fix yet)

In [ ]:
import re
fmt = raw.invoice_date.map(lambda s: "dd/mm/yyyy" if "/" in s else "yyyy-mm-dd" if re.match(r"\d{4}-", s) else "d Mon yyyy")
print("Date formats:\n", fmt.value_counts().to_string(), "\n")
print("Duplicate invoice numbers:\n", raw[raw.invoice_no.duplicated(keep=False)].to_string(), "\n")
print("Credit notes:\n", raw[raw.invoice_no.str.startswith("CN")].to_string(), "\n")
print("Customer names:\n", raw.customer.value_counts().sort_index().to_string())

## 2. Clean, printing the outstanding total after each step
Decisions: dates are UK day-first; remove the exact duplicate; credit notes are negative; standardise 'Harrington and Cole Ltd'.

In [ ]:
from datetime import datetime

def total(d, label):
    print(f"{label:38s} rows {len(d):4d}   outstanding £{(d.amount - d.amount_paid).sum():>12,.2f}")

df = raw.copy(); total(df, "Raw")

def parse(s):
    for f in ("%d/%m/%Y", "%Y-%m-%d", "%d %b %Y"):
        try:
            return datetime.strptime(s, f)
        except ValueError:
            pass
    raise ValueError(f"Unrecognised date: {s}")
df["invoice_date"] = df.invoice_date.map(parse); total(df, "After parsing dates (day-first)")

df = df.drop_duplicates(subset="invoice_no", keep="first"); total(df, "After removing duplicate invoice")

cn = df.invoice_no.str.startswith("CN")
df.loc[cn, "amount"] = -df.loc[cn, "amount"].abs(); total(df, "After making credit notes negative")

df["customer"] = df.customer.replace({"Harrington and Cole Ltd": "Harrington & Cole Ltd"}); total(df, "After standardising customer names")

## 3. Age the balances at 31 March 2026

In [ ]:
YEAR_END = pd.Timestamp("2026-03-31")
df["outstanding"] = df.amount - df.amount_paid
df["days"] = (YEAR_END - df.invoice_date).dt.days
open_items = df[df.outstanding != 0].copy()
open_items["bucket"] = pd.cut(open_items.days, bins=[-1, 30, 60, 90, 10_000], labels=["0-30", "31-60", "61-90", "90+"])

aged = open_items.pivot_table(index="customer", columns="bucket", values="outstanding", aggfunc="sum", fill_value=0, observed=False)
aged["Total"] = aged.sum(axis=1)
aged.loc["Total"] = aged.sum()
aged.round(2)

## 4. Reconcile — aged report total must equal the cleaned ledger

In [ ]:
diff = round(aged.loc["Total", "Total"] - df.outstanding.sum(), 2)
print(f"Aged report £{aged.loc['Total','Total']:,.2f}  vs ledger £{df.outstanding.sum():,.2f}  difference £{diff:,.2f}")
assert diff == 0, "Aged report does not reconcile"

## 5. Provision matrix (IFRS 9 simplified approach) and journal
Rates are illustrative, set by the tutor. Apply them to **bucket totals**, not the gross ledger.

In [ ]:
rates = {"0-30": 0.01, "31-60": 0.03, "61-90": 0.10, "90+": 0.40}
opening_allowance = 24_000.00

matrix = pd.DataFrame({"gross": aged.loc["Total", list(rates)].astype(float), "rate": pd.Series(rates)})
matrix["allowance"] = matrix.gross * matrix.rate
print(matrix.round(2).to_string(), "\n")

closing = round(matrix.allowance.sum(), 2)
movement = round(closing - opening_allowance, 2)
print(f"Required closing allowance  £{closing:,.2f}")
print(f"Opening allowance           £{opening_allowance:,.2f}")
print(f"Movement to journal         £{movement:,.2f}\n")

if movement >= 0:
    dr, cr = "Impairment loss on trade receivables (P&L)", "Allowance for credit losses (SFP)"
else:
    dr, cr = "Allowance for credit losses (SFP)", "Impairment loss on trade receivables (P&L)"
print("Journal, 31 March 2026")
print(f"  Dr {dr:46s} £{abs(movement):,.2f}")
print(f"  Cr {cr:46s} £{abs(movement):,.2f}")

## Discussion
- Which problem would a tool have fixed *silently* if you had asked it to 'clean the data'?
- If the date parser assumed US month-first, which invoices would move bucket?
- Ironbridge Interiors and Dunmore Farms dominate the 90+ bucket. Is a flat 40% right for them, or do they need a specific assessment?